# ResNet — Deep Residual Learning for Image Recognition

Implementation of the residual block concept from He et al. (2015), applied to MNIST digit classification.

## Dataset

MNIST is used here as a lightweight stand-in for the paper's original ImageNet dataset. It consists of 60,000 training and 10,000 test grayscale images of handwritten digits (0–9) at 28×28 pixels. Images are normalized to zero mean and unit variance using MNIST's dataset-wide statistics.

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim

batch_size = 64
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # normalize(mean, std) for MNIST
])

train_dataset = datasets.MNIST(root='../dataset/mnist/', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=batch_size)
test_dataset = datasets.MNIST(root='../dataset/mnist/', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=batch_size)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Residual Block

**Paper:** [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385) (He et al., 2015)

The central idea of ResNet is the **residual (skip) connection**. Instead of learning a direct mapping $H(x)$, each block learns the residual $F(x) = H(x) - x$, so the output becomes:

$$\text{output} = F(x) + x$$

This makes it much easier to train very deep networks because:
- Gradients can flow directly through the skip connection during backpropagation, avoiding vanishing gradients
- If a block is not needed, it can learn $F(x) \approx 0$ and become an identity — the network never gets worse by adding more layers

Each block follows the pattern: **Conv → BN → ReLU → Conv → BN → (add skip) → ReLU**

BatchNorm is applied before the addition so the residual branch is normalized before being combined with the identity shortcut.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        y = F.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        return F.relu(x + y)  # skip connection: add input x to residual y

## Network Architecture

This is a simplified ResNet adapted for MNIST (28×28 grayscale), not the full ImageNet-scale architecture from the paper.

```
Input (1×28×28)
  → Conv5×5 (1→16) + ReLU + MaxPool2×2     → (16×12×12)
  → ResidualBlock(16)                        → (16×12×12)
  → Conv5×5 (16→32) + ReLU + MaxPool2×2    → (32×4×4)
  → ResidualBlock(32)                        → (32×4×4)
  → Flatten                                  → (512,)
  → Linear(512→10)                           → class scores
```

Note: the `ResidualBlock` here keeps the same number of channels in and out, so no projection shortcut is needed. In the full paper, when the channel count doubles (e.g. 64→128), a 1×1 conv is used on the shortcut to match dimensions.

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=5)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=5)

        self.rblock1 = ResidualBlock(16)
        self.rblock2 = ResidualBlock(32)

        self.mp = nn.MaxPool2d(2)
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        in_size = x.size(0)

        x = self.mp(F.relu(self.conv1(x)))
        x = self.rblock1(x)
        x = self.mp(F.relu(self.conv2(x)))
        x = self.rblock2(x)

        x = x.view(in_size, -1)
        x = self.fc(x)
        return x

model = Net().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

## Training

In [ ]:
def train(epoch):
    running_loss = 0.0
    for batch_idx, data in enumerate(train_loader, 0):
        inputs, target = data
        inputs, target = inputs.to(device), target.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if batch_idx % 300 == 299:
            print('[%d, %5d] loss: %.3f' % (epoch+1, batch_idx+1, running_loss/300))
            running_loss = 0.0


def test():
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            images, labels = data
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print('accuracy on test set: %d %% ' % (100*correct/total))


for epoch in range(10):
    train(epoch)
    test()